# Plantellect - EfficientNetV2B1 (frozen head, then fine-tune)

Balanced candidate: direct upgrade from EfficientNetB0, about 8M parameters, fast on CPU.

 the backbone and training gets a second stage that unfreezes the top 50 backbone layers.

It also writes `preprocessing.json` next to the model, which the backend reads so it can switch models safely if server what to use different models.

Outputs go to `plantellect/model/efficientnetv2_b1/` so the runs never overwrite each other or `model/main`. To publish a winner straight to `main`, set `PUBLISH_TO_MAIN = True` in Cell 2 (or copy the folder contents over).

Runtime: *Runtime > Change runtime type > T4 GPU*.

### Cell 1 - imports

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import random
import csv
import time
import shutil
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from PIL import Image
from google.colab import drive

### Cell 2 - Settings

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

RUN_NAME      = "efficientnetv2_b1"
MODEL_ARCH    = "EfficientNetV2B1"
BUILDER       = tf.keras.applications.EfficientNetV2B1
INPUT_SHAPE   = (240, 240, 3)      # native size for this backbone
BATCH_SIZE    = 32
DROPOUT       = 0.5
LABEL_SMOOTHING = 0.0              # 0.0 = identical loss to the B0 run

# How images are resized, in training AND at serving. "nearest" is what flow_from_directory
# NOTE: Options: nearest, bilinear, bicubic, lanczos, box, hamming.
RESIZE_METHOD = "nearest"

# Stage 1: frozen backbone and train head only
EPOCHS        = 50
LEARNING_RATE = 1e-4
# Stage 2: fine-tune the top UNFREEZE_LAYERS backbone layers; BatchNorm stays frozen
FT_EPOCHS        = 20
FT_LEARNING_RATE = 1e-5
UNFREEZE_LAYERS  = 50

AUGMENTATION = {"rotation": 20, "zoom": 0.2, "horizontal_flip": True,
                "brightness": 0.2, "contrast": 0.2}
# "brightness" is applied by the generator in cell 4; "contrast" is applied by a
# RandomContrast layer in cell 5, because ImageDataGenerator has no contrast control.

EARLY_STOPPING_PATIENCE = 10
REDUCE_LR_PATIENCE      = 5

# The saved model is rebuilt in fp32 in cell 9.
MIXED_PRECISION = True
if MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")

MODEL_NAME     = "plantellect_model"
MODEL_FILENAME = MODEL_NAME + ".keras"

PUBLISH_TO_MAIN = False            # NOTE: True -> write into plantellect/model/main

### Cell 3 - LOADING THE DATA

Images are copied from Drive to Colab's local disk once, because `ImageDataGenerator` re-reads every file each epoch and Drive is slow. Set `COPY_LOCAL = False` to read straight from Drive.

In [ ]:
drive.mount('/content/drive')
SRC_DIR  = "/content/drive/MyDrive/plantellect/plantellect_training"
OUT_ROOT = "/content/drive/MyDrive/plantellect/model"
OUT_DIR  = os.path.join(OUT_ROOT, "main" if PUBLISH_TO_MAIN else RUN_NAME)
os.makedirs(OUT_DIR, exist_ok=True)
MODEL_PATH = os.path.join(OUT_DIR, MODEL_FILENAME)

COPY_LOCAL = True


def sync_local(src, dst):
    # Copies files that are missing or whose size differs from Drive. Safe to re-run, so an
    # interrupted earlier copy (half-written files) is repaired instead of silently reused.
    copied = 0
    for root, _, files in os.walk(src):
        out = os.path.join(dst, os.path.relpath(root, src))
        os.makedirs(out, exist_ok=True)
        for f in files:
            s_, d_ = os.path.join(root, f), os.path.join(out, f)
            if not os.path.exists(d_) or os.path.getsize(d_) != os.path.getsize(s_):
                shutil.copy2(s_, d_)
                copied += 1
    return copied


if COPY_LOCAL:
    DATA_DIR = "/content/plantellect_training"
    print("synced from Drive:", sync_local(SRC_DIR, DATA_DIR), "file(s) copied")
else:
    DATA_DIR = SRC_DIR

train_dir = os.path.join(DATA_DIR, "train")
val_dir   = os.path.join(DATA_DIR, "val")
test_dir  = os.path.join(DATA_DIR, "test")

# Must match the extension whitelist inside DirectoryIterator, or trainImages lies.
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".ppm", ".tif", ".tiff")


def class_to_scientific(folder: str) -> str:
    # NOTE: Renaming the directory names by removing the underscore '_' 'Ocotea_acutifolia' or 'Ocotea  acutifolia' -> 'Ocotea acutifolia'
    return "_".join(folder.split()).replace("_", " ").strip()


def list_classes(root: str):
    # Directories only - a stray .DS_Store would become a phantom class.
    return sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))


def count_images(directory: str) -> int:
    # Counts what DirectoryIterator will actually load, recursing like it does.
    return sum(1 for _, _, files in os.walk(directory)
                 for f in files if f.lower().endswith(IMG_EXT))


class_names = list_classes(train_dir)
NUM_CLASSES = len(class_names)
print(f"out: {OUT_DIR}\nin:  {DATA_DIR}\n{NUM_CLASSES} classes")

assert set(class_names) == set(list_classes(val_dir)), \
    "train/ and val/ hold different class names - index shifts would silently mislabel everything"
assert set(class_names) == set(list_classes(test_dir)), \
    "train/ and test/ hold different class names"

counts = {n: count_images(os.path.join(train_dir, n)) for n in class_names}
sizes = sorted(counts.values())
print(f"train per class: min {sizes[0]}, median {sizes[len(sizes) // 2]}, max {sizes[-1]}")
thin = [n for n, c in counts.items() if c < 15]
if thin:
    print("!! under 15 train images:", thin, "- these will make confident wrong predictions")

for name, d in (("val", val_dir), ("test", test_dir)):
    print(f"{name}: {sum(count_images(os.path.join(d, n)) for n in class_names)} images")

### Cell 3b - DECODE CHECK (run before the generators)

Opens and fully decodes every image in train/val/test. `Image.verify()` is not enough, because it passes many truncated files. Each bad file is handled like this:

1. **Good copy in Drive** -> the local file was damaged in transit, so it is re-copied. No data is lost.
2. **Also bad in Drive** -> moved to `/content/quarantine` (never deleted) and added to a shared list, `model/excluded_files.json`. Every notebook applies that list, so all runs train and test on exactly the same files and their scores stay comparable.

Only the local copy is touched, never Drive. This cell refuses to modify anything when `COPY_LOCAL = False`.
this also safely make the run don't get errors from corrupted files for validation, and testing that degrade the training due to curruption.

In [ ]:
EXCLUDE_MANIFEST = os.path.join(OUT_ROOT, "excluded_files.json")
QUARANTINE_DIR   = "/content/quarantine"


def decodes(path):
    try:
        if os.path.getsize(path) == 0:
            return False, "empty file"
        with Image.open(path) as im:
            im.load()                         # full decode; raises on truncated data
        return True, ""
    except Exception as e:
        return False, repr(e)[:110]


def rel(p):
    return os.path.relpath(p, DATA_DIR)


def quarantine(p):
    dst = os.path.join(QUARANTINE_DIR, rel(p))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.move(p, dst)


excluded = json.load(open(EXCLUDE_MANIFEST, encoding="utf-8")) if os.path.exists(EXCLUDE_MANIFEST) else []

# apply the shared exclusion list (identical data for every run)
if COPY_LOCAL:
    for r in excluded:
        if os.path.exists(os.path.join(DATA_DIR, r)):
            quarantine(os.path.join(DATA_DIR, r))

# decode everything that is left
checked, bad = 0, []
for split in ("train", "val", "test"):
    for root, _, files in os.walk(os.path.join(DATA_DIR, split)):
        for f in files:
            if f.lower().endswith(IMG_EXT):
                p = os.path.join(root, f); checked += 1
                ok, why = decodes(p)
                if not ok:
                    bad.append((p, why))
print(f"decoded {checked} images, {len(bad)} failed, {len(excluded)} already on the exclusion list")

if bad and not COPY_LOCAL:
    for p, why in bad:
        print("  CORRUPT:", rel(p), "-", why)
    raise RuntimeError("Corrupt images in Drive. Fix them there, or set COPY_LOCAL = True so this cell can repair the copy.")

repaired, newly_excluded = [], []
for p, why in bad:
    src = os.path.join(SRC_DIR, rel(p))
    if os.path.exists(src) and decodes(src)[0]:
        shutil.copy2(src, p)
        if decodes(p)[0]:
            repaired.append(rel(p)); continue
    quarantine(p); newly_excluded.append((rel(p), why))

if repaired:
    print(f"\nrepaired from Drive ({len(repaired)}): the local copy was damaged, Drive is fine")
    for r in repaired[:15]: print("  ", r)
if newly_excluded:
    print(f"\ncorrupt in Drive too ({len(newly_excluded)}): quarantined, added to the exclusion list")
    for r, why in newly_excluded[:15]: print("  ", r, "-", why)
    os.makedirs(OUT_ROOT, exist_ok=True)
    json.dump(sorted(set(excluded) | {r for r, _ in newly_excluded}),
              open(EXCLUDE_MANIFEST, "w", encoding="utf-8"), indent=2)

# counts were computed in cell 3 - recompute them from what is actually left
counts = {n: count_images(os.path.join(train_dir, n)) for n in class_names}
sizes = sorted(counts.values())
print(f"\ntrain per class: min {sizes[0]}, median {sizes[len(sizes) // 2]}, max {sizes[-1]}")
for name, d in (("train", train_dir), ("val", val_dir), ("test", test_dir)):
    print(f"{name}: {sum(count_images(os.path.join(d, n)) for n in class_names)} images")

### Cell 4 - GENERATORS

In [ ]:
# The model takes float32 pixels in [0, 255]: the backbone's normalisation is inside the
# graph, so nothing out here scales. No preprocess_input anywhere.

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=AUGMENTATION["rotation"],
    zoom_range=AUGMENTATION["zoom"],
    horizontal_flip=AUGMENTATION["horizontal_flip"],
    brightness_range=(1 - AUGMENTATION["brightness"], 1 + AUGMENTATION["brightness"]),
    # NO contrast_range: ImageDataGenerator has no such argument. Contrast is a
    # RandomContrast layer in cell 5.
)
eval_datagen = tf.keras.preprocessing.image.ImageDataGenerator()


def gen(d, g, shuffle=True, seed=None):
    return g.flow_from_directory(
        d, target_size=INPUT_SHAPE[:2], batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=shuffle, seed=seed,
        interpolation=RESIZE_METHOD,
    )


train_generator = gen(train_dir, train_datagen, shuffle=True, seed=SEED)
val_generator = gen(val_dir, eval_datagen, shuffle=False, seed=SEED)
# shuffle=False is REQUIRED for val/test: .classes is in dataset order, so y_pred from
# predict() only lines up with y_true when the iterator is not permuted.
test_generator = gen(test_dir, eval_datagen, shuffle=False)

CLASS_TO_ID = {n: i for i, n in enumerate(class_names)}
ID_TO_CLASS = {i: class_to_scientific(n) for n, i in CLASS_TO_ID.items()}

assert len(CLASS_TO_ID) == NUM_CLASSES
assert train_generator.num_classes == NUM_CLASSES
assert list(train_generator.class_indices) == class_names, \
    "class_indices order differs from class_names - every label would be shifted"
print("index 0 ->", ID_TO_CLASS[0], "| last ->", ID_TO_CLASS[NUM_CLASSES - 1])

### Cell 4b - DATASET INTEGRITY CHECK

It remove images that it natively unable to read

In [ ]:
# flow_from_directory() only reads png/jpg/jpeg/bmp/ppm/tif/tiff. Everything else -
# .heic, .webp, .jfif, .gif, .avif, .DS_Store - is skipped SILENTLY, so a class can train
# on fewer images than its folder holds and nothing tells you.

WHITELIST = IMG_EXT          # mirrors DirectoryIterator.white_list_formats


def find_unsupported(root, names):
    # Files the iterator will never open, grouped by extension.
    by_ext = {}
    for n in names:
        for _, _, files in os.walk(os.path.join(root, n)):
            for f in files:
                if not f.lower().endswith(WHITELIST):
                    by_ext.setdefault(f.rsplit(".", 1)[-1].lower(), []).append(
                        os.path.join(n, f))
    return by_ext


for split, root in (("train", train_dir), ("val", val_dir), ("test", test_dir)):
    found = find_unsupported(root, class_names)
    if found:
        print(f"\n!! {split}: files the loader will silently skip")
        for ext, files in sorted(found.items()):
            print(f"   .{ext:<8} x{len(files):<4} e.g. {files[0]}")
    else:
        print(f"{split}: every file has a supported extension")

assert sum(counts.values()) == train_generator.samples, (
    f"train/ holds {sum(counts.values())} whitelisted images but the generator will load "
    f"only {train_generator.samples} - see the report above.")
print(f"\nwill load - train: {train_generator.samples} | val: {val_generator.samples} | "
      f"test: {test_generator.samples}")

### Cell 5 - loading model

`RandomContrast` is the first layer, active only during training (identity at predict time). The backbone is called with `training=False`, so its BatchNorm and any drop-path stay in inference mode in both stages.

In [ ]:
def build_model(pretrained=True, augment=True):
    base = BUILDER(include_top=False,
                   weights="imagenet" if pretrained else None,
                   input_shape=INPUT_SHAPE, pooling="avg")
    # feature extraction only
    base.trainable = False          

    inputs = tf.keras.Input(shape=INPUT_SHAPE)
    x = inputs
    if augment:
        x = tf.keras.layers.RandomContrast(AUGMENTATION["contrast"], seed=SEED)(x)
    x = base(x, training=False)
    x = tf.keras.layers.Dense(512, activation="relu")(x)
    x = tf.keras.layers.Dropout(DROPOUT)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return tf.keras.Model(inputs, outputs, name=MODEL_NAME), base


def compile_model(m, lr):
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
              metrics=["accuracy"])


def n_trainable(m):
    return int(sum(np.prod(v.shape) for v in m.trainable_variables))


model, base = build_model()
compile_model(model, LEARNING_RATE)
model.summary()
print("backbone layers:", len(base.layers), "| trainable params (stage 1):", f"{n_trainable(model):,}")

### Cell 6 - Training loop (stage 1 head, stage 2 fine-tune)

In [ ]:
total = sum(counts[n] for n in class_names)
class_weight = {i: total / (NUM_CLASSES * counts[n]) for n, i in CLASS_TO_ID.items()}


def cbs():
    return [tf.keras.callbacks.EarlyStopping(patience=EARLY_STOPPING_PATIENCE,
                                             restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=REDUCE_LR_PATIENCE,
                                                 min_lr=1e-7)]


# Stage 1: frozen backbone and traing the Head only
h1 = model.fit(train_generator, epochs=EPOCHS, validation_data=val_generator,
               class_weight=class_weight, callbacks=cbs())
FT_START = len(h1.epoch)          # epoch where fine-tuning begins (for the plots)

# Stage 2: unfreeze the top layers, keep BatchNorm frozen
base.trainable = True
for layer in base.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False
for layer in base.layers[-UNFREEZE_LAYERS:]:
    layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)
compile_model(model, FT_LEARNING_RATE)      # must recompile after changing trainable
print("trainable params (stage 2):", f"{n_trainable(model):,}")

ckpt = tf.keras.callbacks.ModelCheckpoint(os.path.join(OUT_DIR, MODEL_NAME + "_ckpt.keras"),
                                          monitor="val_loss", save_best_only=True)
h2 = model.fit(train_generator, initial_epoch=FT_START, epochs=FT_START + FT_EPOCHS,
               validation_data=val_generator, class_weight=class_weight,
               callbacks=cbs() + [ckpt])

keys = set(h1.history) & set(h2.history)
history = {k: [float(v) for v in h1.history[k] + h2.history[k]] for k in keys}

### Cell 7 - Model Testing (validation + test)

In [ ]:
val_loss,  val_acc  = model.evaluate(val_generator,  verbose=0)
test_loss, test_acc = model.evaluate(test_generator, verbose=0)
print(f"Validation  loss {val_loss:.4f} | accuracy {val_acc:.4f}")
print(f"Test        loss {test_loss:.4f} | accuracy {test_acc:.4f}")

val_probs  = model.predict(val_generator,  verbose=0)
test_probs = model.predict(test_generator, verbose=0)
# NOTE: dataset order - shuffle=False in cell 4
y_val  = val_generator.classes           
y_true = test_generator.classes
y_pred = np.argmax(test_probs, axis=1)
assert len(y_val) == len(val_probs) and len(y_true) == len(test_probs)

assert list(test_generator.class_indices) == class_names
labels = [class_to_scientific(n) for n in test_generator.class_indices.keys()]
assert len(labels) == NUM_CLASSES

top3     = float(np.mean([t in np.argsort(p)[-3:] for t, p in zip(y_true, test_probs)]))
macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
print(f"Test top-3 {top3:.4f} | macro-F1 {macro_f1:.4f}")

idx = list(range(NUM_CLASSES))
test_report = classification_report(y_true, y_pred, labels=idx, target_names=labels, zero_division=0)
val_report  = classification_report(y_val, np.argmax(val_probs, axis=1), labels=idx,
                                    target_names=labels, zero_division=0)
print(test_report)
open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w", encoding="utf-8").write(test_report)
open(os.path.join(OUT_DIR, "val_classification_report.txt"),  "w", encoding="utf-8").write(val_report)

cm = confusion_matrix(y_true, y_pred, labels=idx)
plt.figure(figsize=(14, 12))
plt.imshow(cm, cmap="bwr"); plt.title(f"{MODEL_ARCH} - confusion matrix (test)"); plt.colorbar()
plt.xticks(idx, labels, rotation=90, fontsize=6)
plt.yticks(idx, labels, fontsize=6)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
print("Most confused pairs (true -> predicted):")
for flat in np.argsort(-off, axis=None)[:10]:
    t, p = np.unravel_index(flat, off.shape)
    if off[t, p] > 0:
        print(f"  {off[t, p]:>3}x  {labels[t]}  ->  {labels[p]}")

# .filepaths is in the same order as .classes, so it is the only correct way to recover
# which file a misclassified index refers to.
conf  = test_probs.max(axis=1)
wrong = np.where(y_pred != y_true)[0]
print(f"Misclassified: {len(wrong)} / {len(y_true)}")
filepaths = test_generator.filepaths

with open(os.path.join(OUT_DIR, "misclassified.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["file", "true", "predicted", "confidence"])
    for i in wrong[np.argsort(-conf[wrong])]:           # most confidently wrong first
        w.writerow([filepaths[i], labels[y_true[i]], labels[y_pred[i]], f"{conf[i]:.3f}"])

if len(wrong):
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for ax, i in zip(axes.flat, wrong[:10]):
        ax.imshow(plt.imread(filepaths[i]))             # original file - nothing to undo
        ax.set_title(f"True: {labels[y_true[i]]}\nPred: {labels[y_pred[i]]} ({conf[i]:.0%})", fontsize=7)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "misclassified.png"), dpi=150); plt.show()

`misclassified.csv` lists every wrong test image, most confidently wrong first. Those are the best candidates for mislabelled or near-duplicate images.
best to review to make sure that model trained well

### Cell 8 - Plot training history

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history["loss"], label="Training Loss")
ax1.plot(history["val_loss"], label="Validation Loss")
ax1.axvline(FT_START - 0.5, color="gray", ls="--", label="fine-tune starts")
ax1.set_title(f"{MODEL_ARCH} - Model Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(history["accuracy"], label="Training Accuracy")
ax2.plot(history["val_accuracy"], label="Validation Accuracy")
ax2.axvline(FT_START - 0.5, color="gray", ls="--", label="fine-tune starts")
ax2.set_title(f"{MODEL_ARCH} - Model Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "history.png"), dpi=150)
plt.show()

best = int(np.argmin(history["val_loss"]))
print(f"best epoch {best + 1}: val_loss {history['val_loss'][best]:.4f}, "
      f"val_acc {history['val_accuracy'][best]:.4f}")

### Cell 9 - Saving the Model

The model that gets saved is a clean fp32 copy: no `RandomContrast` (it is identity at inference/server anyway) and no fp16 policy. A model saved with an fp16 policy runs slowly on CPU. Weights are copied layer by layer from the trained model.

In [ ]:
tf.keras.mixed_precision.set_global_policy("float32")
export_model, _ = build_model(pretrained=False, augment=False)


def weighted(m):
    return [l for l in m.layers if isinstance(l, (tf.keras.Model, tf.keras.layers.Dense))]


for dst, src in zip(weighted(export_model), weighted(model)):
    dst.set_weights(src.get_weights())

xb = test_generator[0][0]
diff = float(np.abs(model.predict(xb, verbose=0) - export_model.predict(xb, verbose=0)).max())
print(f"fp32 export vs trained model, max probability difference: {diff:.5f}")
assert diff < 0.05, "export copy does not match the trained model"

export_model.save(MODEL_PATH)

# classes.json maps the model's output index -> scientific name.
with open(os.path.join(OUT_DIR, "classes.json"), "w", encoding="utf-8") as f:
    json.dump({str(i): {"scientific": ID_TO_CLASS[i], "trainImages": counts[class_names[i]]}
               for i in range(NUM_CLASSES)},
              f, indent=2, ensure_ascii=False)

# preprocessing.json: what the backend must do to an uploaded photo before predict() 
# NOTE: only if the server run in multiple models like Current Project Plantellect for Model Comparison.
preprocessing = {
    "schema_version": 1,
    "model": MODEL_ARCH,
    "model_file": MODEL_FILENAME,
    "height": INPUT_SHAPE[0], "width": INPUT_SHAPE[1], "channels": 3,
    "color_mode": "rgb",
    "resize": {"method": RESIZE_METHOD, "keep_aspect_ratio": False},
    # pixel = (pixel * rescale - mean) / std, applied to 0-255 values. the tensorflow only have Py as front end the backend still an C++
    # Identity on purpose: the backbone normalises inside the model. Never divide by 255.
    "rescale": 1.0, "mean": [0.0, 0.0, 0.0], "std": [1.0, 1.0, 1.0],
}
with open(os.path.join(OUT_DIR, "preprocessing.json"), "w", encoding="utf-8") as f:
    json.dump(preprocessing, f, indent=2)

# saving for documentation
with open(os.path.join(OUT_DIR, "history.json"), "w") as f:
    json.dump({"fine_tune_start_epoch": FT_START, **history}, f, indent=2)

metrics = {
    "model_file": MODEL_FILENAME, "arch": MODEL_ARCH, "input_size": list(INPUT_SHAPE[:2]),
    "preprocessing": "none - raw 0-255 pixels, normalisation is inside the model",
    "params": int(export_model.count_params()),
    "val_loss": float(val_loss), "val_accuracy": float(val_acc),
    "test_loss": float(test_loss), "test_accuracy": float(test_acc),
    "test_top3": top3, "test_macro_f1": macro_f1,
    "misclassified": int(len(wrong)), "test_images": int(len(y_true)),
    "best_epoch": best + 1, "epochs_run": len(history["loss"]),
}
with open(os.path.join(OUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("saved:", sorted(os.listdir(OUT_DIR)))

### Cell 10 - artifact verification

In [ ]:
with open(os.path.join(OUT_DIR, "classes.json"), encoding="utf-8") as f:
    served = json.load(f)

reloaded = tf.keras.models.load_model(MODEL_PATH)

assert len(served) == reloaded.output_shape[-1], (
    f"classes.json has {len(served)} entries but the model predicts {reloaded.output_shape[-1]} "
    "- different training runs. Every prediction would be confidently wrong.")
assert len(served) == model.output_shape[-1]
assert not any("_" in v["scientific"] for v in served.values()), "underscore translation was skipped"
assert all(v["scientific"] == v["scientific"].strip() for v in served.values())

rdiff = float(np.abs(reloaded.predict(xb, verbose=0) - export_model.predict(xb, verbose=0)).max())
assert rdiff < 1e-4, f"reloaded file differs from the in-memory model ({rdiff})"

# train/serve parity: the serving recipe in preprocessing.json must reproduce, pixel for
# pixel, what the generator fed the model during training.
from PIL import Image
with open(os.path.join(OUT_DIR, "preprocessing.json"), encoding="utf-8") as f:
    pre = json.load(f)
assert (pre["height"], pre["width"]) == tuple(reloaded.input_shape[1:3]), \
    "preprocessing.json size does not match the saved model's input"


def serve_preprocess(path):
    img = Image.open(path)
    if img.mode != "RGB":
        img = img.convert("RGB")
    img = img.resize((pre["width"], pre["height"]),
                     getattr(Image.Resampling, pre["resize"]["method"].upper()))
    arr = np.asarray(img, dtype=np.float32)
    return (arr * pre["rescale"] - np.array(pre["mean"])) / np.array(pre["std"])


served_batch = np.stack([serve_preprocess(p) for p in test_generator.filepaths[:len(xb)]])
pix_diff = float(np.abs(served_batch - xb).max())
print(f"serving vs training pixels, max difference: {pix_diff} (want 0)")
assert pix_diff <= 1.0, "serving preprocessing differs from training - predictions would be skewed"

x, _ = next(iter(eval_datagen.flow_from_directory(test_dir, target_size=INPUT_SHAPE[:2], batch_size=1)))
print("sample range:", float(x.min()), "to", float(x.max()),
      "(want 0.0 to 255.0 - the model's stem does the normalising, nothing else may)")
print("OK - artifacts consistent")